In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

df = pd.read_csv("../dataset/monthly_spending.csv", parse_dates=["month"])
df = df.sort_values(["user_id", "category", "month"]).reset_index(drop=True)
g = df.groupby(["user_id", "category"])["amount"]

df["avg12"] = g.transform(lambda s: s.shift(1).rolling(12).mean())
df["avg6"]  = g.transform(lambda s: s.shift(1).rolling(6).mean())
df["avg3"]  = g.transform(lambda s: s.shift(1).rolling(3).mean())
df["std6"]  = g.transform(lambda s: s.shift(1).rolling(6).std())

for k in (1, 2, 3):
    df[f"lag{k}_r"] = g.shift(k) / df["avg12"]      # spend k months ago, relative to normal
df["lag12_r"]   = g.shift(12) / df["avg12"]         # same month last year, relative to normal
df["avg3_r"]    = df["avg3"] / df["avg12"]          # recent momentum
df["avg6_r"]    = df["avg6"] / df["avg12"]
df["vol_r"]     = df["std6"] / df["avg12"]          # how jumpy this series is
df["log_level"] = np.log(df["avg12"])               # size of the series
df["cal_month"] = df["month"].dt.month              # captures seasonality

CATS = sorted(df["category"].unique())
df["cat_code"] = df["category"].map({c: i for i, c in enumerate(CATS)})
df["target"] = df["amount"] / df["avg12"]           # what we predict: next month / usual

FEATURES = ["lag1_r", "lag2_r", "lag3_r", "lag12_r", "avg3_r", "avg6_r",
            "vol_r", "log_level", "cal_month", "cat_code"]
CATEGORICAL = [f in ("cal_month", "cat_code") for f in FEATURES]

In [2]:
data = df.dropna(subset=FEATURES + ["target"]).copy()
train = data[data["month"] < "2026-05-01"]
test  = data[data["month"] >= "2026-05-01"].copy()
print(len(train), "train rows,", len(test), "test rows")

model = HistGradientBoostingRegressor(
    loss="absolute_error", categorical_features=CATEGORICAL, random_state=42
)
model.fit(train[FEATURES], train["target"])

test["pred"] = model.predict(test[FEATURES]) * test["avg12"]   # ratio back to rupees

5940 train rows, 1980 test rows


In [3]:
def wape(y, p):
    return np.abs(y - p).sum() / y.sum()

print("Model WAPE:        ", round(wape(test["amount"], test["pred"]), 4))
print("12-month avg WAPE: ", round(wape(test["amount"], test["avg12"]), 4))
print("Model MAE (INR):   ", round(np.abs(test["amount"] - test["pred"]).mean()))

Model WAPE:         0.1659
12-month avg WAPE:  0.183
Model MAE (INR):    1080


In [4]:
test.groupby("category").apply(
    lambda d: pd.Series({"model": wape(d["amount"], d["pred"]),
                         "avg_12m": wape(d["amount"], d["avg12"])})
).round(3)

,model,avg_12m
category,,
Bills & Utilities,0.106,0.149
Entertainment,0.187,0.185
Food,0.101,0.107
Groceries,0.081,0.084
Health,0.347,0.343
Other,0.351,0.360
Personal Care,0.231,0.221
Rent,0.002,0.036
Shopping,0.359,0.371


In [5]:
tot = test.groupby(["user_id", "month"])[["amount", "pred", "avg12"]].sum()
print("Model:   ", round(wape(tot["amount"], tot["pred"]), 4))
print("12m avg: ", round(wape(tot["amount"], tot["avg12"]), 4))

Model:    0.0855
12m avg:  0.0856


In [6]:
m_sq = HistGradientBoostingRegressor(categorical_features=CATEGORICAL, random_state=42)
m_sq.fit(train[FEATURES], train["target"])
p_sq = m_sq.predict(test[FEATURES]) * test["avg12"]
print("Squared-error loss WAPE:", round(wape(test["amount"], p_sq), 4))

Squared-error loss WAPE: 0.1855


In [7]:
pi = permutation_importance(model, test[FEATURES], test["target"], n_repeats=5,
                            random_state=42, scoring="neg_mean_absolute_error")
pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).round(4)

cat_code     0.0058
lag12_r      0.0055
vol_r        0.0049
cal_month    0.0042
lag1_r       0.0029
log_level    0.0024
avg6_r       0.0019
avg3_r       0.0017
lag2_r       0.0014
lag3_r       0.0010
dtype: float64

In [8]:
cutoffs = ["2025-08-01", "2025-11-01", "2026-02-01", "2026-05-01"]
rows = []
for c in cutoffs:
    c = pd.Timestamp(c)
    end = c + pd.DateOffset(months=3)
    tr = data[data["month"] < c]
    te = data[(data["month"] >= c) & (data["month"] < end)].copy()

    m = HistGradientBoostingRegressor(
        loss="absolute_error", categorical_features=CATEGORICAL, random_state=42
    ).fit(tr[FEATURES], tr["target"])
    te["pred"] = m.predict(te[FEATURES]) * te["avg12"]

    tot = te.groupby(["user_id", "month"])[["amount", "pred", "avg12"]].sum()
    rows.append({
        "test window starts": c.date(),
        "train rows": len(tr),
        "model": round(wape(te["amount"], te["pred"]), 4),
        "avg_12m": round(wape(te["amount"], te["avg12"]), 4),
        "model (user total)": round(wape(tot["amount"], tot["pred"]), 4),
        "avg_12m (user total)": round(wape(tot["amount"], tot["avg12"]), 4),
    })
pd.DataFrame(rows)

,test window starts,train rows,model,avg_12m,model (user total),avg_12m (user total)
0,2025-08-01,2970,0.1918,0.2086,0.0984,0.0984
1,2025-11-01,3960,0.1658,0.1896,0.0838,0.0954
2,2026-02-01,4950,0.1482,0.1663,0.0544,0.0550
3,2026-05-01,5940,0.1551,0.1778,0.0681,0.0694


In [9]:
train = data[data["month"] < "2026-05-01"]
test  = data[data["month"] >= "2026-05-01"].copy()

quantile_models = {}
for q in (0.1, 0.5, 0.9):
    quantile_models[q] = HistGradientBoostingRegressor(
        loss="quantile", quantile=q, categorical_features=CATEGORICAL, random_state=42
    ).fit(train[FEATURES], train["target"])

ratios = np.sort(np.column_stack([quantile_models[q].predict(test[FEATURES]) for q in (0.1, 0.5, 0.9)]), axis=1)
test["low"], test["mid"], test["high"] = (ratios * test["avg12"].to_numpy()[:, None]).T
test[["user_id", "month", "category", "low", "mid", "high", "amount"]].head(10).round(0)

C:\Users\hp\AppData\Local\Temp\ipykernel_11260\1946041799.py:12: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  test[["user_id", "month", "category", "low", "mid", "high", "amount"]].head(10).round(0)


,user_id,month,category,low,mid,high,amount
30,U01,2026-05-01,Bills & Utilities,2623.0,3445.0,3850.0,4070.0
31,U01,2026-06-01,Bills & Utilities,2705.0,3393.0,4381.0,3556.0
32,U01,2026-07-01,Bills & Utilities,2405.0,2778.0,3236.0,2812.0
33,U01,2026-08-01,Bills & Utilities,2490.0,2870.0,3185.0,3125.0
34,U01,2026-09-01,Bills & Utilities,2495.0,2939.0,3401.0,2829.0
35,U01,2026-10-01,Bills & Utilities,2525.0,2881.0,3135.0,2338.0
66,U01,2026-05-01,Entertainment,1102.0,1419.0,1958.0,2139.0
67,U01,2026-06-01,Entertainment,1226.0,1760.0,2210.0,1816.0
68,U01,2026-07-01,Entertainment,1094.0,1528.0,2035.0,1453.0
69,U01,2026-08-01,Entertainment,1114.0,1484.0,2007.0,1100.0


In [10]:
inside = (test["amount"] >= test["low"]) & (test["amount"] <= test["high"])
print("Actual inside range:", round(inside.mean(), 3))
print("Below low:", round((test["amount"] < test["low"]).mean(), 3))
print("Above high:", round((test["amount"] > test["high"]).mean(), 3))
print("Median forecast WAPE:", round(wape(test["amount"], test["mid"]), 4))

Actual inside range: 0.759
Below low: 0.106
Above high: 0.135
Median forecast WAPE: 0.1658


In [11]:
test.assign(inside=inside).groupby("category")["inside"].mean().round(2)

category
Bills & Utilities    0.72
Entertainment        0.83
Food                 0.73
Groceries            0.83
Health               0.71
Other                0.73
Personal Care        0.72
Rent                 0.87
Shopping             0.76
Transport            0.73
Travel               0.71
Name: inside, dtype: float64